In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
tennis_data_path = "datos_partidos_tennis-data/"
# Crea una lista con todos los archivos de la ruta que acaban en .csv
files = glob.glob(os.path.join(tennis_data_path, "*.csv"))
tournaments_list = []

for file in files:
    # Lee cada archivo de la lista
    tournament = pd.read_csv(file, encoding='latin1')
    # Todos las columnas en mayúsculas para evitar errores
    tournament.columns = tournament.columns.str.upper().str.strip()
    tournaments_list.append(tournament)

In [3]:
# Junta todos los torneos en un único dataframe
df = pd.concat(tournaments_list, axis=0, ignore_index=True)

In [4]:
len(tournaments_list)

1617

In [5]:
df

,ATP,LOCATION,TOURNAMENT,DATE,SERIES,COURT,SURFACE,ROUND,BEST OF,WINNER,...,BFEW,BFEL,UNNAMED: 38,UNNAMED: 39,ATP\tLOCATION\tTOURNAMENT\tDATE\tSERIES\tCOURT\tSURFACE\tROUND\tBEST OF\tWINNER\tLOSER\tWRANK\tLRANK\tWPTS\tLPTS\tW1\tL1\tW2\tL2\tW3\tL3\tW4\tL4\tW5\tL5\tWSETS\tLSETS\tCOMMENT\tB365W\tB365L\tPSW\tPSL\tMAXW\tMAXL\tAVGW\tAVGL,;;;;;;;;;,UNNAMED: 34,UNNAMED: 35,UNNAMED: 36,UNNAMED: 37
0,15.0,Acapulco,Abierto Mexicano,26/02/01,International Gold,Outdoor,Clay,1st Round,3.0,Arazi H.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15.0,Acapulco,Abierto Mexicano,26/02/01,International Gold,Outdoor,Clay,1st Round,3.0,Blanco G.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,15.0,Acapulco,Abierto Mexicano,26/02/01,International Gold,Outdoor,Clay,1st Round,3.0,Bruguera S.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15.0,Acapulco,Abierto Mexicano,26/02/01,International Gold,Outdoor,Clay,1st Round,3.0,Canas G.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,15.0,Acapulco,Abierto Mexicano,26/02/01,International Gold,Outdoor,Clay,1st Round,3.0,Costa A.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69241,53.0,Zhuhai,Zhuhai Championships,24/09/2023,ATP250,Outdoor,Hard,Quarterfinals,3.0,Karatsev A.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69242,53.0,Zhuhai,Zhuhai Championships,24/09/2023,ATP250,Outdoor,Hard,Quarterfinals,3.0,Nishioka Y.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69243,53.0,Zhuhai,Zhuhai Championships,25/09/2023,ATP250,Outdoor,Hard,Semifinals,3.0,Khachanov K.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69244,53.0,Zhuhai,Zhuhai Championships,25/09/2023,ATP250,Outdoor,Hard,Semifinals,3.0,Nishioka Y.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Elimina columnas erróneas
valid_columns = [c for c in df.columns if 'UNNAMED' not in c and '\t' not in c]
df = df[valid_columns]
df = df.drop(columns=['Ï»¿ATP',';;;;;;;;;'])

# Ordena los partidos por fecha y elimina los partidos sin fecha
df['DATE'] = pd.to_datetime(df['DATE'], dayfirst=True, errors='coerce', format='mixed')
df = df.dropna(subset=['DATE']).sort_values('DATE')

In [7]:
# Elimina columnas que no son de interés
df = df.drop(columns=['ATP','CBW','CBL','EXW','EXL','LBW','LBL','B&WW','B&WL','GBW','GBL','SBW','SBL','UBW','UBL','IWW','IWL',
                      'SJW','SJL','BFEW','BFEL','MAXW','MAXL','AVGW','AVGL'])

In [8]:
# Arreglamos algunas filas y columnas
df['BEST OF'] = df['BEST OF'].fillna(3.0)

df.at[63692,'WINNER'] = "Baez S."
df.at[63692,'LOSER'] = "Muller A."
df.at[63692,'WRANK'] = 31
df.at[63692,'LRANK'] = 60
df.at[63692,'W1'] = 6
df.at[63692,'L1'] = 2
df.at[63692,'W2'] = 6
df.at[63692,'L2'] = 3
df.at[63692,'WSETS'] = 2
df.at[63692,'LSETS'] = 1
df.at[63692,'COMMENT'] = "Completed"
df.at[63692,'B365W'] = 1.73
df.at[63692,'B365L'] = 2.1

mapping_series = {
    'International': 'ATP 250',
    'International Series': 'ATP 250',
    'ATP250': 'ATP 250',
    'International Gold': 'ATP 500',
    'ATP500': 'ATP 500',
    'Masters': 'Masters 1000',
    'Masters 1000': 'Masters 1000',
    'Grand Slam': 'Grand Slam',
    'Masters Cup': 'ATP Finals'
}
df['SERIES'] = df['SERIES'].str.strip().replace(mapping_series)

df['COMMENT'] = df['COMMENT'].replace('Full Time', 'Completed')
df['COMMENT'] = df['COMMENT'].replace('NSY', 'Completed')
df['COMMENT'] = df['COMMENT'].replace('Sched', 'Completed')
df['COMMENT'] = df['COMMENT'].replace('Awarded', 'Retired')
df['COMMENT'] = df['COMMENT'].replace('Rrtired', 'Retired')

df['LOCATION'] = df['LOCATION'].replace('Dubai ', 'Dubai')
df['LOCATION'] = df['LOCATION'].replace('Estoril ', 'Estoril')
df['LOCATION'] = df['LOCATION'].replace("'s-Hertogenbosch", "s-Hertogenbosch")

df.at[51241,'W2'] = 7
df.at[51241,'L2'] = 6
df.at[51241,'W3'] = 6
df.at[51241,'L3'] = 3
df.at[51241,'WSETS'] = 2
df.at[51241,'LSETS'] = 1

df.at[39999,'COMMENT'] = 'Retired'

df.at[35036,'WSETS'] = 2

df.at[10085,'WSETS'] = 2

In [9]:
# Convierte las columnas WRANK y LRANK en Int64 (permite NaN)
df['WRANK'] = df['WRANK'].astype('Int64')
df['LRANK'] = pd.to_numeric(df['LRANK'], errors='coerce')
df['LRANK'] = df['LRANK'].astype('Int64')

In [10]:
print(f"Dimensiones finales del dataset: {df.shape}")

Dimensiones finales del dataset: (66809, 31)


In [11]:
df.head()

,LOCATION,TOURNAMENT,DATE,SERIES,COURT,SURFACE,ROUND,BEST OF,WINNER,LOSER,...,L5,WSETS,LSETS,COMMENT,B365W,B365L,PSW,PSL,WPTS,LPTS
16803,Doha,Qatar Open,2001-01-01,ATP 250,Outdoor,Hard,2nd Round,3.0,Rios M.,Pozzi G.,...,NaN,2.0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,NaN
16785,Doha,Qatar Open,2001-01-01,ATP 250,Outdoor,Hard,1st Round,3.0,Calatrava A.,Koubek S.,...,NaN,2.0,1.0,Completed,NaN,NaN,NaN,NaN,NaN,NaN
16786,Doha,Qatar Open,2001-01-01,ATP 250,Outdoor,Hard,1st Round,3.0,di Pasquale A.,Zabaleta M.,...,NaN,2.0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,NaN
16787,Doha,Qatar Open,2001-01-01,ATP 250,Outdoor,Hard,1st Round,3.0,Escude N.,El Aynaoui Y.,...,NaN,2.0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,NaN
16788,Doha,Qatar Open,2001-01-01,ATP 250,Outdoor,Hard,1st Round,3.0,Jonsson F.,Sanguinetti D.,...,NaN,2.0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df.to_csv('datos_tennis-data.csv', index=False)